# 04 — Pre-registered statistical analysis

Paired per-question analysis matched on `question_id`: bootstrap 95% CI on the mean difference (10,000 resamples, seed 0), McNemar exact test on hard flips (0/8 ↔ 8/8), paired t-test as a sensitivity check. Positive differences mean the intervention produced more misalignment than the control.

Reads `results_{condition}_{split}.json` from notebook 03; writes `prereg_analysis_all_comparisons.json`. No GPU needed. The output below is the reported result set.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# PRE-REGISTERED STATISTICAL ANALYSIS — hyperstition meta-awareness study
#
# Implements the amended pre-registration exactly:
#   * paired per-question analysis, keyed by question_id (NOT list position)
#   * bootstrap 95% CI on the mean difference (10,000 resamples, fixed seed)
#   * McNemar exact test on hard flips (0/8 <-> 8/8)
#   * paired t-test reported as a SENSITIVITY check only
#
# Reads results_{label}.json files written by the evaluation pipeline.
# ════════════════════════════════════════════════════════════════════

import json, os
import numpy as np
from scipy.stats import binomtest, ttest_rel

OUT_DIR = "/content/drive/MyDrive/meta_awareness_v3"
N_BOOT = 10_000
BOOT_SEED = 0

# Every comparison for the write-up, per split.
# Convention: (intervention, control) -> positive difference means the
# INTERVENTION produced MORE misalignment than the control.
COMPARISONS = [
    # Primary
    ("D", "Eprime"),
    # Original primary (now secondary, reported for completeness)
    ("D", "E"),
    # In-context arm
    ("B", "C"),
    # Decompositions against baseline
    ("D", "A"),
    ("E", "A"),
    ("Eprime", "A"),
    ("Eprime", "E"),
    ("B", "A"),
    ("C", "A"),
]
SPLITS = ["articles", "textbook"]


def load_per_question(label):
    """Return {question_id: per-question misalignment rate (mean of 8 flags)}."""
    path = os.path.join(OUT_DIR, f"results_{label}.json")
    with open(path) as f:
        d = json.load(f)
    out = {}
    for r in d["results"]:
        qid = r["question_id"]
        assert qid not in out, f"duplicate question_id {qid} in {label}"
        assert len(r["flags"]) == 8, f"{label} {qid}: expected 8 flags"
        out[qid] = float(np.mean(r["flags"]))
    return out


def analyse(intervention, control, split):
    li, lc = f"{intervention}_{split}", f"{control}_{split}"
    Y = load_per_question(li)   # intervention
    X = load_per_question(lc)   # control

    qids = sorted(set(X) & set(Y))
    n_missing = len(set(X) ^ set(Y))
    assert n_missing == 0, (
        f"{li} vs {lc}: {n_missing} unmatched question_ids — "
        f"a checkpoint is incomplete; do not report this comparison yet."
    )

    y = np.array([Y[q] for q in qids])
    x = np.array([X[q] for q in qids])
    d = y - x

    # ── Pre-registered primary: bootstrap 95% CI on the mean difference ──
    rng = np.random.default_rng(BOOT_SEED)
    idx = rng.integers(0, len(d), size=(N_BOOT, len(d)))
    boot_means = d[idx].mean(axis=1)
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    sig = "SIGNIFICANT" if (lo > 0 or hi < 0) else "not significant"

    # ── Pre-registered: McNemar exact on hard flips ──
    to_mis = int(np.sum((x == 0) & (y == 1)))   # fully aligned -> fully misaligned
    to_ali = int(np.sum((x == 1) & (y == 0)))
    mcnemar_p = (binomtest(to_mis, to_mis + to_ali, 0.5).pvalue
                 if (to_mis + to_ali) > 0 else None)

    # ── Sensitivity check only: paired t-test ──
    t_stat, t_p = ttest_rel(y, x)

    print(f"\n{li}  minus  {lc}")
    print(f"  n = {len(qids)} paired questions (matched on question_id)")
    print(f"  mean difference: {d.mean():+.4f}")
    print(f"  bootstrap 95% CI [{lo:+.4f}, {hi:+.4f}]  ({N_BOOT} resamples, "
          f"seed {BOOT_SEED})  -> {sig}")
    print(f"  hard flips: {to_mis} toward misaligned, {to_ali} toward aligned"
          + (f"  (McNemar exact p = {mcnemar_p:.4f})" if mcnemar_p is not None
             else "  (McNemar not applicable: zero hard flips)"))
    print(f"  sensitivity check, paired t: t = {t_stat:.3f}, p = {t_p:.2e}")

    return {
        "comparison": f"{li} - {lc}", "n": len(qids),
        "mean_diff": float(d.mean()),
        "boot_ci": [float(lo), float(hi)], "significant": sig == "SIGNIFICANT",
        "hard_flips_to_misaligned": to_mis, "hard_flips_to_aligned": to_ali,
        "mcnemar_p": mcnemar_p,
        "paired_t": float(t_stat), "paired_t_p": float(t_p),
        "n_boot": N_BOOT, "boot_seed": BOOT_SEED,
    }


all_results = []
for split in SPLITS:
    print("\n" + "=" * 64)
    print(f"SPLIT: {split.upper()}")
    print("=" * 64)
    for interv, ctrl in COMPARISONS:
        try:
            all_results.append(analyse(interv, ctrl, split))
        except FileNotFoundError as e:
            print(f"\n[skipped] {interv} vs {ctrl} on {split}: {e}")

out_path = os.path.join(OUT_DIR, "prereg_analysis_all_comparisons.json")
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nSaved machine-readable results to {out_path}")